# 0. Setup & Load Data

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r'C:\Code_AI\CS114-FinalTerm\nba_data\nba_full_dataset.csv')
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

print(f'Dataset gốc: {df.shape}')
print(f'Số trận: {df['GAME_ID'].nunique()}')
print(f'Mùa giải: {sorted(df['SEASON'].unique())}')
print(f'\n Các cột hiện tại:')
print(list(df.columns))

Dataset gốc: (12108, 36)
Số trận: 6054
Mùa giải: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

 Các cột hiện tại:
['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'SEASON', 'IS_HOME', 'WIN', 'REST_DAYS', 'IS_B2B', 'GAMES_PLAYED_SEASON', 'CURRENT_WIN_PCT', 'WIN_STREAK']


# 1. Xử lý vấn đề từ EDA

- `FT_PCT` có 1 missing value -> fill 0
- `REST_DAYS` có outlier 150-200 ngày -> clip ở 10 ngày

In [6]:
df['FT_PCT'] = df['FT_PCT'].fillna(0)

print(f"REST_DAYS trước khi cap: min={df['REST_DAYS'].min()}, max={df['REST_DAYS'].max()}")
df['REST_DAYS'] = df['REST_DAYS'].clip(upper=10)

print(f'\nMissing values: {df.isna().sum()}')

REST_DAYS trước khi cap: min=1.0, max=200.0

Missing values: SEASON_ID              0
TEAM_ID                0
TEAM_ABBREVIATION      0
TEAM_NAME              0
GAME_ID                0
GAME_DATE              0
MATCHUP                0
WL                     0
MIN                    0
PTS                    0
FGM                    0
FGA                    0
FG_PCT                 0
FG3M                   0
FG3A                   0
FG3_PCT                0
FTM                    0
FTA                    0
FT_PCT                 0
OREB                   0
DREB                   0
REB                    0
AST                    0
STL                    0
BLK                    0
TOV                    0
PF                     0
PLUS_MINUS             0
SEASON                 0
IS_HOME                0
WIN                    0
REST_DAYS              0
IS_B2B                 0
GAMES_PLAYED_SEASON    0
CURRENT_WIN_PCT        0
WIN_STREAK             0
dtype: int64


# 2. Rolling Averages

Cách xử lý đầu mùa:
- Khi chưa đủ 5 trận -> dùng expanding mean
- Khi đủ 5 trận -> dùng rolling mean
- Trận đầu tiên mỗi mùa -> NaN -> Loại

In [8]:
# Chọn các features có Cohen's d cao và correlation với WIN từ EDA
ROLLING_FEATURES = ['PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT',
                    'REB', 'OREB', 'AST', 'STL', 'BLK', 'TOV']
ROLLING_WINDOW = 5

print(f'Feature for rolling: {ROLLING_FEATURES}')
print(f'Number of features: {len(ROLLING_FEATURES)}')
print(f'Window size: {ROLLING_WINDOW}')

Feature for rolling: ['PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'REB', 'OREB', 'AST', 'STL', 'BLK', 'TOV']
Number of features: 10
Window size: 5


In [13]:
# Sắp xếp theo đội + thời gian
df = df.sort_values(['TEAM_ID', 'SEASON', 'GAME_DATE']).reset_index(drop=True)

for feat in ROLLING_FEATURES:
    col_name = f'ROLLING_{feat}'

    # 1. shift(1) - Loại trận hiện tại 
    shifted = df.groupby(['TEAM_ID', 'SEASON'])[feat].shift(1)

    # 2. Rolling mean
    rolling_val = shifted.groupby([df['TEAM_ID'], df['SEASON']]).rolling(
        window=ROLLING_WINDOW, min_periods=ROLLING_WINDOW
    ).mean().reset_index(level=[0,1], drop=True)

    # 3. Expanding mean
    expanding_val = shifted.groupby([df['TEAM_ID'], df['SEASON']]).expanding(
        min_periods=1
    ).mean().reset_index(level=[0,1], drop=True)

    df[col_name] = rolling_val.fillna(expanding_val)

print("Ví dụ: 8 trận đầu của GSW mùa 2025-26")
print("-" * 80)
atl_mask = (df['TEAM_ABBREVIATION'] == 'GSW') & (df['SEASON'] == '2025-26')
atl_sample = df[atl_mask].head(8)
print(atl_sample[['GAME_DATE', 'PTS', 'ROLLING_PTS', 'AST', 'ROLLING_AST', 'FG_PCT', 'ROLLING_FG_PCT']].to_string(index=False))

Ví dụ: 8 trận đầu của GSW mùa 2025-26
--------------------------------------------------------------------------------
 GAME_DATE  PTS  ROLLING_PTS  AST  ROLLING_AST  FG_PCT  ROLLING_FG_PCT
2025-10-21  119          NaN   29          NaN   0.487             NaN
2025-10-23  137        119.0   33    29.000000   0.505        0.487000
2025-10-24  119        128.0   24    31.000000   0.438        0.496000
2025-10-27  131        125.0   36    28.666667   0.530        0.476667
2025-10-28   98        126.5   24    30.500000   0.453        0.490000
2025-10-30  110        120.8   23    29.200000   0.424        0.482600
2025-11-01  109        119.0   25    28.000000   0.433        0.470000
2025-11-04  118        113.4   33    26.400000   0.494        0.455600


In [15]:
rolling_cols = [f'ROLLING_{f}' for f in ROLLING_FEATURES]
nan_counts = df[rolling_cols].isnull().sum()
nan_rows = df[rolling_cols].isnull().any(axis=1).sum()

print(f"Số dòng có NaN (trận đầu mỗi mùa): {nan_rows}")
print(f"  = {df['TEAM_ID'].nunique()} đội × {df['SEASON'].nunique()} mùa = {df['TEAM_ID'].nunique() * df['SEASON'].nunique()} dòng")
print(f"  → Chiếm {nan_rows/len(df)*100:.1f}% dataset — sẽ bị loại khi ghép")

Số dòng có NaN (trận đầu mỗi mùa): 150
  = 30 đội × 5 mùa = 150 dòng
  → Chiếm 1.2% dataset — sẽ bị loại khi ghép


# 3. Ghép 2 đội thành 1 dòng

- Tách dataset -> HOME (IS_HOME=1) + AWAY(IS_HOME=0)
- Chọn các cột cần thiết, thêm prefix HOME_ / AWAY_
- Join theo GAME_ID
- Label = HOME_WIN (đội nhà thắng = 1, thua = 0)

In [17]:
team_features = (
    [f'ROLLING_{f}' for f in ROLLING_FEATURES]
    + ['CURRENT_WIN_PCT', 'WIN_STREAK', 'REST_DAYS', 'IS_B2B', 'GAMES_PLAYED_SEASON']
)

game_cols = ['GAME_ID', 'GAME_DATE', 'SEASON']

print(f"Features cho mỗi đội ({len(team_features)} cột):")
for f in team_features:
    print(f"  - {f}")

Features cho mỗi đội (15 cột):
  - ROLLING_PTS
  - ROLLING_FG_PCT
  - ROLLING_FG3_PCT
  - ROLLING_FT_PCT
  - ROLLING_REB
  - ROLLING_OREB
  - ROLLING_AST
  - ROLLING_STL
  - ROLLING_BLK
  - ROLLING_TOV
  - CURRENT_WIN_PCT
  - WIN_STREAK
  - REST_DAYS
  - IS_B2B
  - GAMES_PLAYED_SEASON


In [25]:
# Split HOME / AWAY
home_df = df[df['IS_HOME'] == 1].copy()
away_df = df[df['IS_HOME'] == 0].copy()

print(f"HOME: {len(home_df)} dòng")
print(f"AWAY: {len(away_df)} dòng")

# Rename columns
home_rename = {col: f'HOME_{col}' for col in team_features}
home_rename['TEAM_ABBREVIATION'] = 'HOME_TEAM'
home_rename['WIN'] = 'HOME_WIN'

away_rename = {col: f'AWAY_{col}' for col in team_features}
away_rename['TEAM_ABBREVIATION'] = 'AWAY_TEAM'

home_df = home_df.rename(columns=home_rename)
away_df = away_df.rename(columns=away_rename)

# Chọn cột cần thiết
home_cols = game_cols + ['HOME_TEAM', 'HOME_WIN'] + [f'HOME_{f}' for f in team_features]
away_cols = ['GAME_ID', 'AWAY_TEAM'] + [f'AWAY_{f}' for f in team_features]

home_df = home_df[home_cols]
away_df = away_df[away_cols]

# Merge
merged = home_df.merge(away_df, on='GAME_ID', how='inner')

# Loại bỏ dòng có NaN (trận đầu mùa không có rolling)
before_drop = len(merged)
merged = merged.dropna().reset_index(drop=True)
after_drop = len(merged)

print(f"\nSau khi ghép: {before_drop} trận")
print(f"Sau khi loại NaN: {after_drop} trận (mất {before_drop - after_drop} trận đầu mùa)")
print(f"\nDataset shape: {merged.shape[0]} dòng × {merged.shape[1]} cột")

HOME: 6044 dòng
AWAY: 6064 dòng

Sau khi ghép: 6044 trận
Sau khi loại NaN: 5965 trận (mất 79 trận đầu mùa)

Dataset shape: 5965 dòng × 36 cột


**PHÁT HIỆN VẤN ĐỀ KHI TÁCH HOME / AWAY**
- Số lượng home_df và away_df khác nhau. 
- Đây là các trận cả 2 đội đều ghi IS_HOME = 0 => Neutral


In [26]:
# Loại trận neutral site (cả 2 đội đều IS_HOME=0)
game_home_count = df.groupby('GAME_ID')['IS_HOME'].sum()
neutral_games = game_home_count[game_home_count != 1].index
df = df[~df['GAME_ID'].isin(neutral_games)]

In [29]:
# Split HOME / AWAY
home_df = df[df['IS_HOME'] == 1].copy()
away_df = df[df['IS_HOME'] == 0].copy()

print(f"HOME: {len(home_df)} dòng")
print(f"AWAY: {len(away_df)} dòng")

# Rename columns
home_rename = {col: f'HOME_{col}' for col in team_features}
home_rename['TEAM_ABBREVIATION'] = 'HOME_TEAM'
home_rename['WIN'] = 'HOME_WIN'

away_rename = {col: f'AWAY_{col}' for col in team_features}
away_rename['TEAM_ABBREVIATION'] = 'AWAY_TEAM'

home_df = home_df.rename(columns=home_rename)
away_df = away_df.rename(columns=away_rename)

# Chọn cột cần thiết
home_cols = game_cols + ['HOME_TEAM', 'HOME_WIN'] + [f'HOME_{f}' for f in team_features]
away_cols = ['GAME_ID', 'AWAY_TEAM'] + [f'AWAY_{f}' for f in team_features]

home_df = home_df[home_cols]
away_df = away_df[away_cols]

# Merge
merged = home_df.merge(away_df, on='GAME_ID', how='inner')

# Loại bỏ dòng có NaN (trận đầu mùa không có rolling)
before_drop = len(merged)
merged = merged.dropna().reset_index(drop=True)
after_drop = len(merged)

print(f"\nSau khi ghép: {before_drop} trận")
print(f"Sau khi loại NaN: {after_drop} trận (mất {before_drop - after_drop} trận đầu mùa)")
print(f"\nDataset shape: {merged.shape[0]} dòng × {merged.shape[1]} cột")

HOME: 6044 dòng
AWAY: 6044 dòng

Sau khi ghép: 6044 trận
Sau khi loại NaN: 5965 trận (mất 79 trận đầu mùa)

Dataset shape: 5965 dòng × 36 cột


In [30]:
# 3.7 — Kiểm tra kết quả
print("Ví dụ 3 dòng đầu tiên:")
print("=" * 80)
sample = merged.head(3)
for idx, row in sample.iterrows():
    print(f"\nTrận {idx+1}: {row['HOME_TEAM']} (nhà) vs {row['AWAY_TEAM']} (khách) | {row['GAME_DATE']}")
    print(f"  HOME: rolling_PTS={row['HOME_ROLLING_PTS']:.1f}, WIN_PCT={row['HOME_CURRENT_WIN_PCT']:.3f}, streak={row['HOME_WIN_STREAK']}")
    print(f"  AWAY: rolling_PTS={row['AWAY_ROLLING_PTS']:.1f}, WIN_PCT={row['AWAY_CURRENT_WIN_PCT']:.3f}, streak={row['AWAY_WIN_STREAK']}")
    print(f"  Kết quả: {'HOME thắng' if row['HOME_WIN']==1 else 'AWAY thắng'}")

print(f"\nLabel distribution:")
print(f"  HOME thắng: {merged['HOME_WIN'].sum()} ({merged['HOME_WIN'].mean():.1%})")
print(f"  AWAY thắng: {(merged['HOME_WIN']==0).sum()} ({(merged['HOME_WIN']==0).mean():.1%})")

Ví dụ 3 dòng đầu tiên:

Trận 1: ATL (nhà) vs DET (khách) | 2021-10-25 00:00:00
  HOME: rolling_PTS=104.0, WIN_PCT=0.500, streak=-1
  AWAY: rolling_PTS=85.0, WIN_PCT=0.000, streak=-2
  Kết quả: HOME thắng

Trận 2: ATL (nhà) vs WAS (khách) | 2021-11-01 00:00:00
  HOME: rolling_PTS=104.8, WIN_PCT=0.500, streak=-2
  AWAY: rolling_PTS=115.6, WIN_PCT=0.833, streak=3
  Kết quả: HOME thắng

Trận 3: ATL (nhà) vs UTA (khách) | 2021-11-04 00:00:00
  HOME: rolling_PTS=106.6, WIN_PCT=0.500, streak=-1
  AWAY: rolling_PTS=113.8, WIN_PCT=0.857, streak=2
  Kết quả: AWAY thắng

Label distribution:
  HOME thắng: 3298 (55.3%)
  AWAY thắng: 2667 (44.7%)
